In [1]:
from pathlib import Path
from qqe.GNN.training.datasets import build_loaders, build_loaders_NN

In [2]:
def collect_files_path(
    data_dir: str,
    family: str | None = None,
    backend: str | None = "pennylane",
    target: str | None = "sre",
) -> list[str]:
    """Collects file paths matching the pattern in the given directory."""
    d = Path(data_dir)
    if family is not None:
        paths = sorted((d / f"encoding_data_{target}_{backend}" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / f"encoding_data_{target}_{backend}"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

In [20]:
family_filter = "clifford"

data_paths = collect_files_path("../data/training_data", family=family_filter)
if not data_paths:
    raise RuntimeError("No data paths found.")

print(f"Found {len(data_paths)} data paths.")

Found 35000 data paths.


In [21]:
train_loader, val_loader, test_loader, node_in_dim, global_in_dim, base_dataset = (
    build_loaders(
        data_paths,
        batch_size=50,
        seed=24,
        train_split=0.8,
        val_split=0.15,
        global_feature_variant="binned",
        node_feature_variant=None,
        family_projection=family_filter,
    )
)

In [22]:
print(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

23800 4200 7000


In [23]:
training_data = train_loader.dataset
validation_data = val_loader.dataset
test_data = test_loader.dataset

In [24]:
from collections import Counter

def count_regimes(dataset):
    return Counter(data.meta["regime"] for data in dataset)



In [25]:
train_counts = count_regimes(training_data)
val_counts   = count_regimes(validation_data)
test_counts  = count_regimes(test_data)

print("Train:", train_counts)
print("Val:", val_counts)
print("Test:", test_counts)

Train: Counter({'low': 8179, 'medium': 7276, 'high': 4771, 'zero': 3574})
Val: Counter({'low': 1486, 'medium': 1242, 'high': 833, 'zero': 639})
Test: Counter({'low': 2428, 'medium': 2124, 'high': 1384, 'zero': 1064})


In [26]:
import pandas as pd

def regime_table(train_dataset, val_dataset, test_dataset):
    train = count_regimes(train_dataset)
    val   = count_regimes(val_dataset)
    test  = count_regimes(test_dataset)

    df = pd.DataFrame([train, val, test], index=["train", "val", "test"])
    return df.fillna(0).astype(int)

df_regimes = regime_table(training_data, validation_data, test_data)
print(df_regimes)

        low  high  medium  zero
train  8179  4771    7276  3574
val    1486   833    1242   639
test   2428  1384    2124  1064


In [27]:
df_regimes.div(df_regimes.sum(axis=1), axis=0)

,low,high,medium,zero
train,0.343655,0.200462,0.305714,0.150168
val,0.353810,0.198333,0.295714,0.152143
test,0.346857,0.197714,0.303429,0.152000
